In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch

# Load tokenizer & model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model for regression (reward modeling)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


data = {
    "prompt": ["Explain quantum computing", "What is RLHF?"],
    "chosen": ["Quantum computing uses qubits...", "RLHF means reinforcement learning with feedback..."],
    "rejected": ["It's complicated...", "Not sure about that."]
}
dataset = Dataset.from_dict(data)

# Preprocessing function
def preprocess(example):
    input_text = example["prompt"] + " " + example["chosen"]
    tokenized = tokenizer(
        input_text,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokenized["labels"] = 1.0  
    return tokenized

# Apply preprocessing
tokenized_dataset = dataset.map(preprocess, batched=False)

# PyTorch dataset wrapper
class RewardDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        example = self.dataset[idx]
        return {
            "input_ids": torch.tensor(example["input_ids"]),
            "attention_mask": torch.tensor(example["attention_mask"]),
            "labels": torch.tensor(example["labels"], dtype=torch.float)
        }

    def __len__(self):
        return len(self.dataset)

train_dataset = RewardDataset(tokenized_dataset)

# Training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-reward-model",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Train and save
trainer.train()
trainer.save_model("./tinyllama-reward-model")
tokenizer.save_pretrained("./tinyllama-reward-model")


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at TinyLlama/TinyLlama-1.1B-Chat-v1.0 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2/2 [00:00<00:00, 46.52 examples/s]


Step,Training Loss


SafetensorError: Error while serializing: IoError(Os { code: 112, kind: StorageFull, message: "There is not enough space on the disk." })

In [ ]:
from trl import PPOTrainer, PPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import torch


model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)



class DummyRewardModel(torch.nn.Module):
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        batch_size = input_ids.shape[0]
        # Return random rewards for demonstration
        return torch.nn.utils.rnn.PackedSequence(
            data=torch.randn(batch_size, 1), batch_sizes=torch.tensor([batch_size])
        )

reward_model = DummyRewardModel()

# PPO config
ppo_config = PPOConfig(
    model_name=model_name,
    learning_rate=1e-5,
    batch_size=4,
    forward_batch_size=2
)

# PPO Trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
    reward_model=reward_model
)

# Sample prompts
prompts = ["What is quantum entanglement?", "Explain reinforcement learning"]

# PPO Training Loop
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    response_ids = model.generate(**inputs, max_new_tokens=100)
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    # Compute dummy reward
    with torch.no_grad():
        reward_input = tokenizer(prompt + " " + response, return_tensors="pt", truncation=True, padding=True).to(model.device)
        reward_score = torch.rand(1).item()  

    # PPO optimization step
    ppo_trainer.step([prompt], [response], [reward_score])


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")


In [15]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

prompt = "Explain quantum computing in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output tokens
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated text:\n", generated_text)


Generated text:
 Explain quantum computing in simple terms.


In [8]:
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to(device)


In [ ]:
import copy
ref_model = copy.deepcopy(ppo_model)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False
